# Feature B — Checkpointed multilingual sentiment pipeline

Labels every review as positive/neutral/negative, validates against manually checked real reviews, compares two Gemini models and the star baseline, resumes without relabelling completed reviews, and creates product/category summaries.

In [ ]:
!pip install -q google-genai python-dotenv pyarrow tqdm scikit-learn matplotlib seaborn

In [ ]:
import os,sys
from pathlib import Path
from getpass import getpass
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from google.colab import drive
drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/Amazon Mini Projects/Amazon-Marketplace-Product-Intelligence')
assert PROJECT.exists(); sys.path.insert(0,str(PROJECT))
from analysis.cleaning import load_clean
from analysis.labelling_model2 import GeminiLabeller,resolve_model_name
from analysis.sentiment_checkpointed import prepare_reviews,manual_template,validate_manual_file,label_with_checkpoint,sentiment_summaries
if not os.getenv('GEMINI_API_KEY'): os.environ['GEMINI_API_KEY']=getpass('Gemini API key: ')
CHECKPOINT_DIR=PROJECT/'data'/'checkpoints'/'ReviewSentiment'
OUTPUT_DIR=PROJECT/'data'/'models'/'ReviewSentiment'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

## 1. Preprocessing, EDA, and decisions

The cleaned schema uses `rating` (not `star_rating`). All languages are retained. EDA is performed on all cleaned reviews; model evaluation uses a persistent human-labelled subset.

In [ ]:
products,reviews=load_clean(); df=prepare_reviews(reviews)
print('Reviews:',len(df),'duplicates:',df.review_id.duplicated().sum())
display(df); display(df.isna().mean().mul(100).sort_values(ascending=False).to_frame('missing_percent'))
fig,axes=plt.subplots(1,3,figsize=(18,5))
sns.countplot(data=df,y='language',order=df.language.value_counts().head(15).index,ax=axes[0]); axes[0].set_title('Top review languages')
sns.countplot(data=df,x='rating',ax=axes[1]); axes[1].set_title('Rating distribution')
sns.histplot(df.body_clean.str.len(),bins=50,ax=axes[2]); axes[2].set_title('Review length')
plt.tight_layout(); plt.show()
print('English share:',round(df.language.eq('en').mean(),4)); print(df.star_sentiment.value_counts(dropna=False))

### EDA interpretation

The report found 9,250 reviews: 85.6% English and 14.4% across Spanish, French, Portuguese, German, Italian, Arabic and smaller language groups. Therefore the manual set is stratified by English/non-English and rating-derived sentiment. Native multilingual classification is used to avoid translation cost and sentiment distortion. Ratings are only a baseline—not human truth. Review IDs and body hashes prevent unsafe checkpoint reuse.

In [ ]:
MANUAL_FILE=CHECKPOINT_DIR/'manual_sentiment_ground_truth.csv'
manual=manual_template(df,MANUAL_FILE,n_per_stratum=15)
display(manual)
print('ACTION: open this CSV in Drive and fill manual_sentiment. Do not copy star_sentiment.')
print(MANUAL_FILE)

## 2. Resume here after manual annotation

This cell deliberately stops until real manual labels exist. The CSV is never overwritten, so completed human work survives every rerun.

In [ ]:
gold=validate_manual_file(MANUAL_FILE)
print('Completed human labels:',len(gold)); display(gold.manual_sentiment.value_counts())
MODEL_2=resolve_model_name('gemini-3.5-flash-lite',('gemini-3.1-flash-lite','gemini-2.5-flash-lite'))
MODELS=['gemini-3.6-flash',MODEL_2]
comparison=[]; model_outputs={}
for model_name in MODELS:
    labeller=GeminiLabeller(model_name=model_name,batch_size=10,request_delay=6.5)
    labelled,path=label_with_checkpoint(prepare_reviews(gold),labeller,CHECKPOINT_DIR,prompt_version='sentiment_v2',batch_size=10)
    valid=labelled[labelled.sentiment.isin(['positive','neutral','negative'])]
    if len(valid):
        comparison.append({'model':model_name,'success_rate':len(valid)/len(gold),'accuracy':accuracy_score(valid.manual_sentiment,valid.sentiment),'macro_f1':f1_score(valid.manual_sentiment,valid.sentiment,average='macro')})
        model_outputs[model_name]=labelled
star=gold[gold.star_sentiment.isin(['positive','neutral','negative'])]
comparison.append({'model':'star_baseline','success_rate':len(star)/len(gold),'accuracy':accuracy_score(star.manual_sentiment,star.star_sentiment),'macro_f1':f1_score(star.manual_sentiment,star.star_sentiment,average='macro')})
comparison=pd.DataFrame(comparison).sort_values(['macro_f1','success_rate'],ascending=False); display(comparison)
api_candidates=comparison[comparison.model!='star_baseline']; FINAL_MODEL=api_candidates.iloc[0].model
print('Selected final API model:',FINAL_MODEL)

## 3. Label all reviews with safe checkpointing

Run repeatedly across quota windows. Existing valid labels for the same review body, model, and prompt version are reused; only missing/failed rows are requested again.

In [ ]:
final_labeller=GeminiLabeller(model_name=FINAL_MODEL,batch_size=10,request_delay=6.5)
labelled_all,checkpoint=label_with_checkpoint(df,final_labeller,CHECKPOINT_DIR,prompt_version='sentiment_v2',batch_size=10)
complete=labelled_all.sentiment.isin(['positive','neutral','negative']).sum()
print('Completed:',complete,'/',len(labelled_all),'checkpoint:',checkpoint)
if complete<len(labelled_all): print('Quota/run incomplete: rerun this cell later; completed labels will not be requested again.')

## 4. Product and category summaries

Run after all—or enough—reviews are labelled. Percentages and dominant sentiment are produced for every ASIN and search category.

In [ ]:
usable=labelled_all[labelled_all.sentiment.isin(['positive','neutral','negative'])].copy()
product_summary,category_summary=sentiment_summaries(usable)
product_summary.to_csv(OUTPUT_DIR/'sentiment_by_product.csv',index=False)
category_summary.to_csv(OUTPUT_DIR/'sentiment_by_category.csv',index=False)
usable.to_parquet(OUTPUT_DIR/'reviews_with_sentiment.parquet',index=False)
display(product_summary); display(category_summary)
print('Saved summaries and labelled reviews:',OUTPUT_DIR)